In [ ]:

from google.colab import files

uploaded = files.upload()  # Select ML_Budget_Dataset_WorldBank_Filled(1).xlsx from your machine

Saving ML_Budget_Dataset_WorldBank_Completed.xlsx to ML_Budget_Dataset_WorldBank_Completed.xlsx


In [ ]:

from google.colab import files

uploaded = files.upload()  # Select ML_Budget_Dataset_WorldBank_Filled(1).xlsx from your machine

Saving database_2026(1).xlsx to database_2026(1).xlsx


In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor

!pip install catboost
from catboost import CatBoostRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

# 1. Load Dataset
file_path = "ML_Budget_Dataset_WorldBank_Completed.xlsx"
df = pd.read_excel(file_path, sheet_name="Completed_Dataset")

# 2. Define Features (X)
feature_cols = [
    'population', 'urban_population_percent', 'gdp_usd', 'gdp_per_capita_usd',
    'gdp_growth_percent', 'inflation_percent', 'tax_revenue_usd', 'government_debt_usd',
    'poverty_rate_percent', 'unemployment_rate_percent', 'literacy_rate_percent',
    'life_expectancy_years', 'health_expenditure_usd', 'internet_users_percent',
    'electricity_access_percent', 'co2_emissions', 'corruption_index', 'government_effectiveness'
]

# 3. Target Allocations (Sectors & SDGs)
sector_targets = [
    'defence_usd', 'infrastructure_usd', 'research_and_development_usd',
    'education_usd', 'healthcare_usd', 'agriculture_usd',
    'rural_development_usd', 'social_welfare_freebies_usd', 'energy_usd',
    'transport_usd', 'housing_urban_development_usd', 'water_sanitation_usd',
    'environment_climate_usd', 'digital_it_usd', 'industry_msme_usd'
]

sdg_percent_targets = [
    'sdg1_no_poverty_percent', 'sdg2_zero_hunger_percent', 'sdg3_good_health_percent',
    'sdg4_quality_education_percent', 'sdg6_clean_water_percent', 'sdg7_clean_energy_percent',
    'sdg8_decent_work_percent', 'sdg9_industry_innovation_percent',
    'sdg11_sustainable_cities_percent', 'sdg13_climate_action_percent', 'sdg16_peace_justice_percent'
]

X = df[feature_cols]

# 4. CatBoost Evaluation Function
def evaluate_catboost(X_data, target_cols, dataset_title):
    X_train, X_test, y_train, y_test = train_test_split(
        X_data, df[target_cols], test_size=0.20, random_state=42
    )

    cb_base = CatBoostRegressor(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=42,
        verbose=False
    )

    catboost_model = MultiOutputRegressor(cb_base)
    catboost_model.fit(X_train, y_train)

    y_preds = catboost_model.predict(X_test)
    preds_df = pd.DataFrame(y_preds, columns=target_cols, index=y_test.index)

    results = []
    for target in target_cols:
        mae = mean_absolute_error(y_test[target], preds_df[target])
        rmse = np.sqrt(mean_squared_error(y_test[target], preds_df[target]))
        mape = mean_absolute_percentage_error(y_test[target], preds_df[target]) * 100
        accuracy = max(0, 100 - mape)
        r2 = r2_score(y_test[target], preds_df[target])

        results.append({
            'Target Name': target.replace('_usd', '').replace('_percent', '').replace('_', ' ').title(),
            'MAE': mae,
            'RMSE': rmse,
            'MAPE (%)': mape,
            'Accuracy (%)': accuracy,
            'R² Score': r2
        })

    res_df = pd.DataFrame(results)
    print(f"\n=================== {dataset_title} ===================")
    print(res_df.to_string(index=False))

    print(f"\n--- CatBoost Overall Averages ---")
    print(f"Macro Avg Accuracy Rate : {res_df['Accuracy (%)'].mean():.2f}%")
    print(f"Macro Avg MAPE          : {res_df['MAPE (%)'].mean():.2f}%")
    print(f"Macro Avg R² Score      : {res_df['R² Score'].mean():.4f}")
    return res_df

# Run CatBoost Evaluation
cb_sector_df = evaluate_catboost(X, sector_targets, "CATBOOST SECTOR ALLOCATIONS")
cb_sdg_df = evaluate_catboost(X, sdg_percent_targets, "CATBOOST SDG BUDGET SHARES")


=================== CATBOOST SECTOR ALLOCATIONS ===================
              Target Name          MAE         RMSE  MAPE (%)  Accuracy (%)  R² Score
                  Defence 5.636349e+09 1.408024e+10 31.760373     68.239627  0.985702
           Infrastructure 2.163253e+09 3.992248e+09 10.588698     89.411302  0.997588
 Research And Development 3.344751e+09 6.176014e+09 95.951654      4.048346  0.995075
                Education 9.195122e+09 1.854112e+10 20.653085     79.346915  0.992060
               Healthcare 1.238542e+10 3.098675e+10 21.002836     78.997164  0.995418
              Agriculture 1.081627e+09 1.996124e+09 10.588698     89.411302  0.997588
        Rural Development 9.464234e+08 1.746609e+09 10.588698     89.411302  0.997588
  Social Welfare Freebies 3.244880e+09 5.988372e+09 10.588698     89.411302  0.997588
                   Energy 1.352033e+09 2.495155e+09 10.588698     89.411302  0.997588
                Transport 1.622440e+09 2.994186e+09 10.588698     89.41